## Recreation of Hunstville Ignition Energy Paper

In [185]:
import cantera as ct
import CoolProp as cp
import numpy as np
import pandas as pd

In [186]:
PSI2PA = 6894.76 # psi to pa
R2K = 0.555556 # Rankine to Kelvin
BTU2J = 1055.06 # BTU to J
LBM2KG = 0.453592 # lbm to kg

In [187]:
# Torch
fuel_torch = "propane"
ox_torch = "oxygen"
mech_torch = "gri30.yaml" # torch reaction mechanism
OF_torch = 1.15

# MCA
fuel = "ethanol"
ox = "oxygen"
mech_MCA = "Reduced_highMech.yaml" # MCA reaction mechanism
OF_MCA = 1.14
mdot_main = 0.5985


T1_f = 285
T2_f = 365 + 273
T1_ox = 90
T2_ox = 365 + 273
Pc = 278 * PSI2PA


### Find Change in enthalpy required to reach Autoigniton Temperature for MCA Propellants

In [188]:
h1_ox = cp.CoolProp.PropsSI("H", "T", T1_ox, "P", Pc, ox)
h2_ox = cp.CoolProp.PropsSI("H", "T", T2_ox, "P", Pc, ox)
h1_f = cp.CoolProp.PropsSI("H", "T", T1_f, "P", Pc, fuel)
h2_f = cp.CoolProp.PropsSI("H", "T", T2_f, "P", Pc, fuel)

dh_f = h2_f - h1_f
dh_ox = h2_ox - h1_ox

print(dh_f* 10**-6, dh_ox* 10**-6)

1.6008278609119213 0.730729152933289


In [189]:
dh_mix = (1/(OF_MCA+1)) * dh_f + (OF_MCA/(OF_MCA+1)) * dh_ox # J/kg
print(f"Change in enthalpy: {dh_mix * 10**-6:0.3f} [MJ/kg]")

Change in enthalpy: 1.137 [MJ/kg]


In [190]:
# function to get masss averaged chemical potential of mixture 

def mass_avg_chem_pot(mixture):
    mu = mixture.chemical_potentials      # J/kmol
    W  = mixture.molecular_weights        # kg/kmol
    Y  = mixture.Y                        # mass fractions
    return np.dot(Y, mu / W)              # J/kg mixture

In [191]:
# find difference in chemical potentials in torch reaction to find energy released from reaction
torch_rxn = ct.Solution(mech_torch)
torch_rxn.TPY = 298, ct.one_atm, {"O2": OF_torch/(OF_torch+1), "C3H8":(1/(OF_torch+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(torch_rxn)
torch_rxn.equilibrate("HP")
post_rxn = mass_avg_chem_pot(torch_rxn)

print(f"Torch Flame Temp: {torch_rxn.T:0.1f}")
combustion_energy_torch = np.abs(post_rxn - pre_rxn)
print(f"{combustion_energy_torch* 10**-6:0.3f}")

Torch Flame Temp: 1525.9
22.582


In [192]:
# same process for MCA Combustion energy
# find difference in chemical potentials in torch reaction to find energy released from reaction
MCA_rxn = ct.Solution(mech_MCA)
MCA_rxn.TPY = 298, Pc, {"O2": OF_MCA/(OF_MCA+1), "C2H5OH":(1/(OF_MCA+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(MCA_rxn)
MCA_rxn.equilibrate("HP")
print(MCA_rxn.T)
post_rxn = mass_avg_chem_pot(MCA_rxn)

combustion_energy_MCA = np.abs(post_rxn - pre_rxn)
print(f"{combustion_energy_MCA* 10**-6:0.3f}")

2898.380906441231
35.548


/var/folders/gb/6k0pj_b16wsgr1x344s7m8gr0000gn/T/ipykernel_79400/3440174243.py:3: UserWarning: TroeRate::setFalloffCoeffs: Unexpected parameter value T2=0. Omitting exp(T2/T) term from falloff expression. To suppress this warning, remove value for T2 from the input file. In the unlikely case that the exp(T2/T) term should be included with T2 effectively equal to 0, set T2 to a sufficiently small value (for example, T2 < 1e-16).
  MCA_rxn = ct.Solution(mech_MCA)


In [193]:
# Compute percent of MCA that needs to ignite for chain reaction
x = dh_mix/combustion_energy_MCA
P_ig = mdot_main * x * dh_mix
P_ig * 10**-3

np.float64(21.77778454665299)

In [194]:
mdot_torch = 2 * P_ig / combustion_energy_torch
print(f"Igniter mdot: {mdot_torch*10**3:0.3f}")

Igniter mdot: 1.929


### Find combustion energy of torch reaction

In [195]:
# set up reactor network for torch reaction

sol_spark_products = ct.Solution(mech_torch)
sol_spark_products.equilibrate("HP")

sol_torch = ct.Solution(mech_torch)
sol_torch.TPY = 298, Pc, {"O2": OF_torch/(OF_torch+1), "C3H8":(1/(OF_torch+1)) } 

res_upstream = ct.Reservoir(sol_torch, clone=True)
res_downstream = ct.Reservoir(sol_torch, clone=True)

wsr_torch = ct.ConstPressureReactor(sol_torch, energy="on", clone=True)

mfc_torch_in = ct.MassFlowController(upstream=res_upstream, downstream=wsr_torch, mdot=1)
#mfc_torch_out = ct.MassFlowController(upstream=wsr_torch, downstream=res_downstream, mdot=1)
pc_torch_out = ct.PressureController(upstream=wsr_torch, downstream=res_downstream, primary=mfc_torch_in, K=1e-5)

sim = ct.ReactorNet([wsr_torch])

In [196]:
sim.solve_steady()

In [197]:
wsr_torch.phase.T

297.99999999999994

### Tabled for now: Injecting pilot flame into a wsr
#### -> can be either (likely both) torch combustion products -> torch ignition or torch combustion products -> MCA 
#### -> may be able to incorporate flame kernel radius into this

In [198]:
sol_spark_products = ct.Solution(mech_torch)
sol_spark_products.TPY = 298, Pc, {"O2": OF_torch/(OF_torch+1), "C3H8":(1/(OF_torch+1)) } 
print(sol_spark_products.T)
sol_spark_products.equilibrate("HP")
print(sol_spark_products.T)
print(sol_spark_products.chemical_potentials)

298.0
1545.992071870177
[-2.10203145e+08 -1.05101572e+08 -3.82571508e+08 -7.65143015e+08
 -4.87673080e+08 -5.92774652e+08 -8.70244588e+08 -9.75346160e+08
 -4.64965377e+07 -1.51598110e+08 -2.56699682e+08 -2.56699682e+08
 -3.61801254e+08 -4.66902827e+08 -4.29068045e+08 -8.11639553e+08
 -5.34169618e+08 -6.39271190e+08 -7.44372762e+08 -7.44372762e+08
 -8.49474334e+08 -1.98094648e+08 -3.03196220e+08 -4.08297792e+08
 -5.13399364e+08 -6.18500937e+08 -7.23602509e+08 -5.80666155e+08
 -6.85767728e+08 -6.85767728e+08 -8.63277961e+09 -8.80338942e+09
 -8.99970900e+09 -9.24348312e+09 -8.99191403e+09 -9.11664931e+09
 -9.23695922e+09 -9.16113447e+09 -9.12827938e+09 -8.75596748e+09
 -9.07843522e+09 -9.00655053e+09 -8.84465772e+09 -9.12258642e+09
 -9.30017890e+09 -9.40639002e+09 -9.13141292e+09 -9.17693212e+09
 -9.10766709e+09 -8.75200619e+08 -9.80302191e+08 -7.90869300e+08
 -8.95970872e+08]
